# Dynamic Bayesian Network â€” Machine-Failure Prediction from Alarm Data

I build a Dynamic Bayesian Network that predicts if a
machine will transition to a **Failure** state in the next time window, given its
current state and the alarm behaviour observed in the current window.

$$P(\text{State}_{t+1} = \text{Failure} \mid \text{State}_t,\ \text{alarm features at } t)$$

### 1. What is a Bayesian Network (BN)?
A Bayesian Network is a probabilistic graphical model that represents a set of random variables and their conditional dependencies through a Directed Acyclic Graph (DAG). Grounded in Bayes' Theorem, it lets me compactly represent the joint probability distribution of an entire system by exploiting the conditional independencies between variables. Bayesian Networks are widely used for diagnostic and predictive reasoning under uncertainty.

### 2. What is a Dynamic Bayesian Network (DBN)?
A Dynamic Bayesian Network extends the traditional Bayesian Network to model temporal, sequential, or time-series data. While a standard BN captures a static "snapshot" of a system, a DBN connects multiple BNs sequentially across discrete time slices. It assumes the Markov property, the state of the system at time $t$ depends only on the state at time $t-1$, which keeps the model computationally tractable.

### 3. How are temporal dependencies represented in a DBN?
Temporal dependencies are modelled with **transition edges** (also called inter-slice edges) that cross from one time slice to the next. For example, a directed edge connects a node at time $t$ (e.g. `Machine_State_t`) to a node at time $t+1$ (`Machine_State_t+1`). These cross-slice arrows explicitly capture how past observations and history influence the future state.

### 4. What are Nodes, Directed Edges, and Conditional Probability Tables (CPTs)?
- **Nodes:** the building blocks of the graph, each representing a random variable. In this exercise the nodes represent variables such as *Alarm Count*, *Alarm Duration*, and the *Machine State* (Running / Failure).
- **Directed Edges:** arrows connecting nodes to describe influence or direct conditional dependency. An arrow from node $A$ to node $B$ means that $B$ is probabilistically conditioned on $A$.
- **Conditional Probability Tables (CPTs):** tables attached to each node that quantify the probability distribution of that node given every possible combination of its parents' states.

### Import Libraries

I import all the libraries at first that are needed for the exercise:
- **pandas** and **numpy** â€” for data loading and manipulation
- **pgmpy** â€” the Bayesian Network library, I use three parts of it:
  - `DynamicBayesianNetwork` â€” to declare and display the DBN structure
  - `BayesianNetwork` + `BayesianEstimator` â€” to learn the CPTs from data
  - `VariableElimination` â€” for exact probabilistic inference
- **warnings** â€” just to suppress noisy output

In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import DynamicBayesianNetwork as DBN
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
except ImportError:
    from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

C:\Users\psorosan\Remaintain\PR_TUW_2025_ReMAIntAIn\Sorosanszki\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


C:\Users\psorosan\Remaintain\PR_TUW_2025_ReMAIntAIn\Sorosanszki\.venv\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## Data Preprocessing & Feature Extraction

This part covers **Steps 0 and 1** of the exercise:

1. Load the dataset and compute each alarm's **active duration** in seconds
   (`end_alarm - start_alarm`)
2. Define **discretisation functions** that convert raw numbers into categories
   as specified in the exercise sheet
3. Select the **3 most frequent alarm IDs** as A1, A2, A3 (the exercise assumes
   three monitored alarms, I pick the most frequent since the dataset has 94)
4. Aggregate alarm events into one row per `time_window`, computing each alarm's
   count and total duration, then discretising both.
5. Generate **transitions**, pairs of consecutive windows (t â†’ t+1), which is
   what the DBN learns from.

### Load & compute duration

Parse the timestamps and derive `duration_seconds` for each alarm record.

In [2]:
# Load dataset
df = pd.read_csv('dataset_exercise.csv', delimiter=';')
df['start_alarm'] = pd.to_datetime(df['start_alarm'])
df['end_alarm']   = pd.to_datetime(df['end_alarm'])
df['duration_seconds'] = (df['end_alarm'] - df['start_alarm']).dt.total_seconds()

print(f"Loaded {len(df):,} rows | {df['time_window'].nunique()} time windows | "
      f"{df['alarm_id'].nunique()} unique alarm IDs")

Loaded 4,307 rows | 329 time windows | 94 unique alarm IDs


### Discretisation functions

Converting raw numbers to categories â€” required because Bayesian Networks
work with discrete values, not continuous ones.

| Count | Category | | Duration (s) | Category |
|-------|----------|-|--------------|----------|
| 0 | None | | 0 | None |
| 1â€“2 | Low | | 1â€“30 | Short |
| 3â€“5 | Medium | | 31â€“300 | Medium |
| >5 | High | | >300 | Long |

In [3]:
# Discretisation functions
def alarm_count(count):
    if count == 0:     return 'None'
    elif count <= 2:   return 'Low'
    elif count <= 5:   return 'Medium'
    else:              return 'High'

def alarm_duration(duration):
    if duration == 0:       return 'None'
    elif duration <= 30:    return 'Short'
    elif duration <= 300:   return 'Medium'
    else:                   return 'Long'

### Select the three monitored alarms

The exercise says three alarms A1, A2, A3 are monitored. I pick the
3 most frequently occurring alarm IDs in the dataset.

In [4]:
top_alarms = df['alarm_id'].value_counts().head(3).index.tolist()
for i, aid in enumerate(top_alarms, 1):
    print(f"A{i} = {aid}  ({(df['alarm_id'] == aid).sum()} occurrences)")

A1 = 156701909  (606 occurrences)
A2 = 156701901  (594 occurrences)
A3 = 156701902  (589 occurrences)


### Build one feature row per time window

For each `time_window`:
- Take the **majority machine state**
- Count how many times each alarm fired â†’ `alarm_count()`
- Sum the total active duration of each alarm â†’ `alarm_duration()`

The result is one row per window with 7 columns:
`State`, `A1_Count`, `A1_Duration`, `A2_Count`, `A2_Duration`, `A3_Count`, `A3_Duration`

In [5]:
def build_window_features(df, top_alarms):
    processed_data = []
    for w, window_data in df.groupby('time_window'):
        state = window_data['machine_state'].mode().iloc[0]
        row   = {'time_window': w, 'State': state}
        for i, alarm in enumerate(top_alarms, start=1):
            sub = window_data[window_data['alarm_id'] == alarm]
            row[f'A{i}_Count']    = alarm_count(len(sub))
            row[f'A{i}_Duration'] = alarm_duration(sub['duration_seconds'].sum())
        processed_data.append(row)
    return (pd.DataFrame(processed_data)
              .sort_values('time_window')
              .reset_index(drop=True))

df_processed = build_window_features(df, top_alarms)

print(f"{len(df_processed)} windows built")
print("State distribution:", df_processed['State'].value_counts().to_dict())
df_processed.head()

329 windows built
State distribution: {'Running': 207, 'Failure': 122}


,time_window,State,A1_Count,A1_Duration,A2_Count,A2_Duration,A3_Count,A3_Duration
0,1,Failure,Medium,Long,Medium,Long,Medium,Long
1,2,Failure,None,None,None,None,None,None
2,4,Failure,Medium,Long,Medium,Long,Medium,Long
3,5,Running,Medium,Medium,Medium,Medium,Medium,Medium
4,6,Running,Medium,Medium,Medium,Medium,Medium,Medium


### Generate training transitions (Step 3)

A DBN learns from pairs of consecutive windows: situation at time t
paired with the state at time t+1.

I use `shift(-1)` to pair each window with the next, but I have to filter out the gaps in the `df['time_window'] ` column, windows whose IDs are not exactly 1 apart are not consecutive.


In [6]:
df_processed['State_next']   = df_processed['State'].shift(-1)
df_processed['next_window']  = df_processed['time_window'].shift(-1)

df_transitions = df_processed[
    df_processed['next_window'] == df_processed['time_window'] + 1
].dropna().copy()

print(f"{len(df_transitions)} valid consecutive transitions")
print("Next-state distribution:", df_transitions['State_next'].value_counts().to_dict())
df_transitions.head()

195 valid consecutive transitions
Next-state distribution: {'Running': 119, 'Failure': 76}


,time_window,State,A1_Count,A1_Duration,A2_Count,A2_Duration,A3_Count,A3_Duration,State_next,next_window
0,1,Failure,Medium,Long,Medium,Long,Medium,Long,Failure,2.0
2,4,Failure,Medium,Long,Medium,Long,Medium,Long,Running,5.0
3,5,Running,Medium,Medium,Medium,Medium,Medium,Medium,Running,6.0
6,20,Failure,Medium,Long,Medium,Long,Medium,Long,Failure,21.0
7,21,Failure,Medium,Long,Medium,Long,Medium,Long,Running,22.0


### Step 4 â€” Declare the DBN Structure

I declare the network using pgmpy's `DynamicBayesianNetwork`.
Nodes are tuples `(variable, time_slice)`, so `('State', 0)` is the
current state and `('State', 1)` is the next state.

Every current alarm feature and the current machine state point into the next state â€” these are the inter-slice edges that define the temporal
dependencies:

```
('State', 0)       â”€â”€â–º  ('State', 1)
('A1_Count', 0)    â”€â”€â–º  ('State', 1)
('A1_Duration', 0) â”€â”€â–º  ('State', 1)
('A2_Count', 0)    â”€â”€â–º  ('State', 1)
('A2_Duration', 0) â”€â”€â–º  ('State', 1)
('A3_Count', 0)    â”€â”€â–º  ('State', 1)
('A3_Duration', 0) â”€â”€â–º  ('State', 1)
```

In [7]:
# Declare the DBN structure (Step 4)
dbn = DBN()
edges = [
    (('State', 0),       ('State', 1)),
    (('A1_Count', 0),    ('State', 1)),
    (('A1_Duration', 0), ('State', 1)),
    (('A2_Count', 0),    ('State', 1)),
    (('A2_Duration', 0), ('State', 1)),
    (('A3_Count', 0),    ('State', 1)),
    (('A3_Duration', 0), ('State', 1)),
]
dbn.add_edges_from(edges)

print("DBN structure defined. Inter-slice edges:")
for (src, t0), (tgt, t1) in dbn.get_inter_edges():
    print(f"  ({src}, {t0})  ->  ({tgt}, {t1})")

DBN structure defined. Inter-slice edges:
  (State, 0)  ->  (State, 1)
  (A1_Count, 0)  ->  (State, 1)
  (A1_Duration, 0)  ->  (State, 1)
  (A2_Count, 0)  ->  (State, 1)
  (A2_Duration, 0)  ->  (State, 1)
  (A3_Count, 0)  ->  (State, 1)
  (A3_Duration, 0)  ->  (State, 1)


### Step 5 â€” Learn the Conditional Probability Tables (CPTs)

pgmpy's `DynamicBayesianNetwork.fit()` currently only supports MLE and
requires all variables at both time slices, this is a constraint that does not fit to the data because the alarm features only exist at time t

I therefore use an equivalent flat **`BayesianNetwork`** with the same
edges and train it with a **Bayesian (BDeu) prior**:
- MLE would assign probability 0 to unseen alarm combinations â†’ inference fails
- BDeu smoothing gives every combination a small non-zero probability â†’ robust

`equivalent_sample_size=5` controls smoothing strength.

I also do a **chronological 75/25 train/test split** â€” first 75% for
training, last 25% for testing. I do not mix future data for training.

In [8]:
Features = ['A1_Count', 'A1_Duration', 'A2_Count',
            'A2_Duration', 'A3_Count', 'A3_Duration']

Count_categs    = ['None', 'Low', 'Medium', 'High']
Duration_categs = ['None', 'Short', 'Medium', 'Long']
States        = ['Running', 'Failure']

# Build flat BayesianNetwork (same edges as the DBN above)
parent_cols = ['State_t'] + [f'{c}_t' for c in Features]
model = BayesianNetwork([(p, 'State_t1') for p in parent_cols])

# Rename transition columns to t / t+1 format for the model
train_df = df_transitions.rename(columns={
    'State': 'State_t', 'State_next': 'State_t1',
    **{c: f'{c}_t' for c in Features}
})


split = int(len(train_df) * 0.75)
train = train_df.iloc[:split].reset_index(drop=True)
test  = train_df.iloc[split:].reset_index(drop=True)
print(f"Train: {len(train)}  |  Test: {len(test)}")

Train: 146  |  Test: 49


In [9]:
# Learn CPTs with Bayesian (BDeu) smoothing
state_names = {'State_t': States, 'State_t1': States}
for c in Features:
    state_names[f'{c}_t'] = Count_categs if c.endswith('Count') else Duration_categs

cols      = parent_cols + ['State_t1']
estimator = BayesianEstimator(model, train[cols], state_names=state_names)
cpds      = estimator.get_parameters(prior_type='BDeu', equivalent_sample_size=5)
model.add_cpds(*cpds)

print("DBN trained successfully! The conditional probability tables (CPTs) have been learned.")
print("Model valid:", model.check_model())

DBN trained successfully! The conditional probability tables (CPTs) have been learned.
Model valid: True


## Step5 â€” Inference & Evaluation

### Inference

I use **Variable Elimination** â€” an exact probabilistic inference algorithm.
Given evidence (current state + alarm features), it computes the full
probability distribution over `State_t+1`.

The `query()` helper returns `P(Failure | evidence)` as a float between 0 and 1.

### Example query

I reproduce the example from Step 5 of the exercise:
$$P(\text{Failure} \mid \text{State}=\text{Running},\ A1=\text{High/Long},\ A2=\text{Medium/Medium},\ A3=\text{Low/Short})$$

### Evaluation

I evaluate on the **held-out test transitions** â€” the last 25% of the data, which the model never saw during training â€” using:
- **Accuracy** â€” overall correct predictions
- **Precision** â€” of all predicted Failures, how many were real?
- **Recall** â€” of all real Failures, how many did I catch?
- **F1** â€” harmonic mean of precision and recall (main metric)
- **Confusion matrix** â€” TP / FP / FN / TN breakdown
- **Baseline** â€” a persistence model that simply predicts "next state = current state". If my model scores higher than this baseline, the alarm features are adding real predictive value beyond just assuming nothing changes.

In [10]:
infer = VariableElimination(model)

def query(state_t, alarm_features):
    """Returns P(State_t+1 = Failure | evidence)."""
    evidence = {'State_t': state_t}
    evidence.update({f'{k}_t': v for k, v in alarm_features.items()})
    try:
        q   = infer.query(['State_t1'], evidence=evidence, show_progress=False)
        idx = q.state_names['State_t1'].index('Failure')
        return float(q.values[idx])
    except Exception:
        return 0.5   # fallback to prior if unseen

In [11]:
# Example query from the exercise
evidence = {
    'A1_Count': 'High',   'A1_Duration': 'Long',
    'A2_Count': 'Medium', 'A2_Duration': 'Medium',
    'A3_Count': 'Low',    'A3_Duration': 'Short',
}
p = query('Running', evidence)
print("Probability of Machine State in the Next Time Window:")
print(f"  P(Failure | State=Running, A1=High/Long, A2=Medium/Medium, A3=Low/Short) = {p:.3f}")
print(f"  P(Running | ...) = {1-p:.3f}")

Probability of Machine State in the Next Time Window:
  P(Failure | State=Running, A1=High/Long, A2=Medium/Medium, A3=Low/Short) = 0.500
  P(Running | ...) = 0.500


In [12]:
# Evaluate on the test set
tp = fp = fn = tn = 0
for _, r in test.iterrows():
    feats  = {c: r[f'{c}_t'] for c in Features}
    p_fail = query(r['State_t'], feats)
    pred   = 'Failure' if p_fail >= 0.5 else 'Running'
    true   = r['State_t1']
    tp += (true == 'Failure' and pred == 'Failure')
    fp += (true == 'Running' and pred == 'Failure')
    fn += (true == 'Failure' and pred == 'Running')
    tn += (true == 'Running' and pred == 'Running')

n    = tp + fp + fn + tn
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
acc  = (tp + tn) / n
base = (test['State_t'] == test['State_t1']).mean()

print("=" * 40)
print("EVALUATION  (positive class = Failure)")
print("=" * 40)
print(f"Accuracy  : {acc:.3f}")
print(f"Precision : {prec:.3f}")
print(f"Recall    : {rec:.3f}")
print(f"F1 Score  : {f1:.3f}")
print(f"Confusion : TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Baseline  : {base:.3f}  (persistence: next = current)")

EVALUATION  (positive class = Failure)
Accuracy  : 0.878
Precision : 0.889
Recall    : 0.889
F1 Score  : 0.889
Confusion : TP=24  FP=3  FN=3  TN=19
Baseline  : 0.816  (persistence: next = current)


### Highest-risk Configurations

Scan all alarm configurations that appeared at least twice in the training
data and rank them by predicted `P(Failure)`.

This gives an interpretable view of what the model learned, which alarm
patterns are most strongly associated with an upcoming failure.

In [13]:
scan = (train.groupby(['State_t'] + [f'{c}_t' for c in Features])
             .size().reset_index(name='n')
             .query('n >= 2'))

ranked = []
for _, r in scan.iterrows():
    feats = {c: r[f'{c}_t'] for c in Features}
    ranked.append((query(r['State_t'], feats), r['State_t'], feats))

rows = []
for p, st, feats in sorted(ranked, key=lambda x: x[0], reverse=True)[:8]:
    rows.append({'State_t': st, **feats, 'P(Failure)': round(p, 3)})

pd.DataFrame(rows)

,State_t,A1_Count,A1_Duration,A2_Count,A2_Duration,A3_Count,A3_Duration,P(Failure)
0,Failure,None,None,None,None,None,None,0.769
1,Failure,Low,Long,Low,Long,Low,Long,0.727
2,Failure,Low,Short,Low,Short,Low,Short,0.667
3,Failure,High,Long,High,Long,High,Long,0.667
4,Failure,Low,Medium,Low,Medium,Low,Medium,0.500
5,Failure,Medium,Medium,Medium,Medium,Medium,Medium,0.500
6,Failure,Medium,Long,Medium,Long,Medium,Long,0.455
7,Running,Low,Long,Low,Long,Low,Long,0.400
